# Bottom-Up 3D Wireframe Swarm Generation (Anchor + Edge Subdivision)

This notebook demonstrates the **Bottom-Up 3D Wireframe Swarm Generation Pipeline**:
- **Zero PyTorch / Point-E dependencies:** Pure geometric math running in $<1$ ms on CPU.
- **Topological Integrity:** Mandatory corner anchors guarantee shape recognizability even at ultra-low drone budgets ($N < 20$).
- **Proportional Edge Subdivision:** Remaining drone budget is distributed proportionally across structural edges according to Euclidean segment length.
- **Zero-WebGL 3D Rendering:** Uses Matplotlib 3D (`mplot3d`) for guaranteed inline 3D visualization in VS Code (independent of browser WebGL hardware support), alongside interactive Plotly HTML exports.

## Cell 1: Imports & Loading `sample_wireframes.json`

Loads the geometric wireframe graph definitions and initializes 3D plotting libraries (Matplotlib 3D + Plotly).

In [ ]:
# ==============================================================================
# Cell 1: Imports & Loading sample_wireframes.json
# ==============================================================================
import os
import json
import numpy as np

# Matplotlib 3D (Renders directly into notebook cell as image - NO WebGL required!)
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Plotly 3D (Interactive rotation & WebGL viewer)
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# Configure Plotly
pio.renderers.default = "notebook+vscode"

# Ensure output directory exists for standalone HTML plots
os.makedirs("./outputs", exist_ok=True)

# Load wireframe graphs
with open("sample_wireframes.json", "r", encoding="utf-8") as f:
    wireframe_db = json.load(f)

print("=" * 68)
print("              LOADED WIREFRAME GEOMETRY DATABASE                      ")
print("=" * 68)
for name, data in wireframe_db.items():
    n_anchors = len(data["anchors"])
    n_edges = len(data["edges"])
    desc = data.get("description", "")
    print(f" • {name:<14} | Anchors (Corners): {n_anchors:2d} | Edges: {n_edges:2d} | {desc}")
print("=" * 68)

## Cell 2: Core Bottom-Up Subdivision Function (`build_swarm_from_graph`)

The algorithm allocates $N$ drones to a 3D wireframe graph:
1. **If $N \le K$ ($K$ anchors):** Uses Farthest Point Sampling (FPS) from the origin-farthest anchor to preserve the widest outer extremities.
2. **If $N > K$:**
   - Locks the first $K$ drones directly to all corner anchors (labeled `"anchor"`).
   - Computes Euclidean edge lengths $L_e = \|v_j - v_i\|_2$.
   - Proportionally allocates the remaining budget $(N - K)$ across edges based on segment length:
     $$\text{exact}_e = \frac{L_e}{\sum L} \cdot (N - K)$$
   - Distributes integer rounding remainders to edges with the largest fractional parts.
   - Generates evenly spaced interior drones along each edge:
     $$P_s = \left(1 - \frac{s}{m + 1}\right) v_i + \left(\frac{s}{m + 1}\right) v_j \quad \text{for } s = 1, \dots, m$$
3. **Helper:** [`compute_min_distance(points)`](#) calculates the minimum pairwise clearance distance $d_{\min}$.

In [ ]:
# ==============================================================================
# Cell 2: Core Bottom-Up Subdivision Function (build_swarm_from_graph)
# ==============================================================================
def build_swarm_from_graph(graph: dict, num_drones: int) -> tuple[np.ndarray, list[str]]:
    """
    Builds a 3D drone swarm formation from a wireframe graph for any target drone count N:
    - If num_drones <= K: FPS selects the widest outer extremities from anchors.
    - If num_drones > K: Locks all K anchors, then proportionally subdivides edges.
    
    Returns:
        points: np.ndarray of shape (num_drones, 3)
        point_types: list of strings ('anchor' or 'edge') of length num_drones
    """
    anchors = np.array(graph["anchors"], dtype=float)
    edges = graph["edges"]
    K = len(anchors)

    # Case 1: Ultra-small swarm (N <= K anchors) -> FPS selects widest anchors
    if num_drones <= K:
        selected_idx = np.zeros(num_drones, dtype=int)
        distances = np.full(K, np.inf)

        # Start with the point farthest from the origin
        selected_idx[0] = np.argmax(np.linalg.norm(anchors, axis=1))

        for i in range(1, num_drones):
            last_pt = anchors[selected_idx[i - 1]]
            dist_to_last = np.sum((anchors - last_pt) ** 2, axis=1)
            distances = np.minimum(distances, dist_to_last)
            selected_idx[i] = np.argmax(distances)

        points = anchors[selected_idx]
        point_types = ["anchor"] * num_drones
        return points, point_types

    # Case 2: Full swarm (N > K anchors) -> Lock all anchors + subdivide edges
    points = [pt for pt in anchors]
    point_types = ["anchor"] * K
    remaining = num_drones - K

    # Compute Euclidean edge lengths
    edge_lengths = np.array([
        np.linalg.norm(anchors[j] - anchors[i]) for i, j in edges
    ], dtype=float)
    total_len = np.sum(edge_lengths)

    # Proportional allocation
    exact = (edge_lengths / (total_len + 1e-8)) * remaining
    drones_per_edge = np.floor(exact).astype(int)
    remainder = remaining - np.sum(drones_per_edge)

    # Distribute integer rounding remainder to edges with highest fractional parts
    if remainder > 0:
        fractional = exact - drones_per_edge
        top_indices = np.argsort(fractional)[-remainder:]
        drones_per_edge[top_indices] += 1

    # Generate evenly spaced interior edge points
    for (i, j), m in zip(edges, drones_per_edge):
        if m > 0:
            for s in range(1, m + 1):
                t = s / (m + 1.0)
                pt = (1.0 - t) * anchors[i] + t * anchors[j]
                points.append(pt)
                point_types.append("edge")

    return np.array(points, dtype=float), point_types


def compute_min_distance(points: np.ndarray) -> float:
    """
    Computes the minimum pairwise Euclidean distance between any two drones:
    d_min = min_{a != b} ||p_a - p_b||_2
    """
    if len(points) < 2:
        return 0.0
    diffs = points[:, None, :] - points[None, :, :]
    dists = np.linalg.norm(diffs, axis=-1)
    np.fill_diagonal(dists, np.inf)
    return float(np.min(dists))


# Quick verification test across all shapes
print("Algorithm Verification:")
for shape_name, g in wireframe_db.items():
    pts, ptypes = build_swarm_from_graph(g, num_drones=30)
    dmin = compute_min_distance(pts)
    print(f" ✓ {shape_name:<14} -> N=30 | Anchors: {ptypes.count('anchor'):2d} | Edges: {ptypes.count('edge'):2d} | d_min: {dmin:.4f}")

## Cell 3: Multi-Scale Drone Count Comparison ($N = 12, 25, 45, 80$)

Renders a 1-row, 4-column **True 3D Perspective Comparison** of `"fighter_jet"`:
- **Gold Markers (`#FFD700`, size=75):** Mandatory corner anchors.
- **Cyan Markers (`#00E5FF`, size=35):** Interpolated structural edge drones.
- **Dashed Lines (`rgba(255, 255, 255, 0.25)`):** Underlying structural wireframe edges.
- **No WebGL Required:** Renders instantly via Matplotlib 3D (`mplot3d`) directly inside VS Code.

In [ ]:
# ==============================================================================
# Cell 3: Multi-Scale Drone Count Comparison (N = 12, 25, 45, 80)
# ==============================================================================
SHAPE_NAME = "fighter_jet"
SCALE_COUNTS = [12, 25, 45, 80]

graph = wireframe_db[SHAPE_NAME]
anchors = np.array(graph["anchors"], dtype=float)
edges = graph["edges"]

# ------------------------------------------------------------------------------
# 1. TRUE 3D MATPLOTLIB PLOT (100% Visible - No WebGL dependency)
# ------------------------------------------------------------------------------
fig_mpl = plt.figure(figsize=(18, 5))
fig_mpl.patch.set_facecolor('#0b0f19')

for idx, N in enumerate(SCALE_COUNTS, start=1):
    ax = fig_mpl.add_subplot(1, 4, idx, projection='3d')
    ax.set_facecolor('#0b0f19')
    points, point_types = build_swarm_from_graph(graph, N)
    d_min = compute_min_distance(points)

    # 1. Wireframe edges (dashed lines)
    for i, j in edges:
        ax.plot(
            [anchors[i, 0], anchors[j, 0]],
            [anchors[i, 1], anchors[j, 1]],
            [anchors[i, 2], anchors[j, 2]],
            color='white', linestyle='--', linewidth=0.8, alpha=0.25
        )

    # 2. Anchor points (Gold)
    anchor_mask = [t == "anchor" for t in point_types]
    if any(anchor_mask):
        apts = points[anchor_mask]
        ax.scatter(apts[:, 0], apts[:, 1], apts[:, 2], c='#FFD700', s=75, edgecolors='white', linewidth=0.7, depthshade=True)

    # 3. Edge drones (Cyan)
    edge_mask = [t == "edge" for t in point_types]
    if any(edge_mask):
        epts = points[edge_mask]
        ax.scatter(epts[:, 0], epts[:, 1], epts[:, 2], c='#00E5FF', s=35, alpha=0.9, depthshade=True)

    ax.set_title(f"N = {N} Drones\nd_min = {d_min:.3f}", color='white', fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel("X", color='#94a3b8', labelpad=4)
    ax.set_ylabel("Y", color='#94a3b8', labelpad=4)
    ax.set_zlabel("Z", color='#94a3b8', labelpad=4)
    ax.tick_params(colors='#94a3b8', labelsize=7)
    ax.set_box_aspect([1, 1, 1])
    ax.view_init(elev=22, azim=45)

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('#1e293b')
    ax.yaxis.pane.set_edgecolor('#1e293b')
    ax.zaxis.pane.set_edgecolor('#1e293b')
    ax.grid(color='#334155', linestyle=':', alpha=0.5)

plt.suptitle(f"3D Multi-Scale Wireframe Swarm: '{SHAPE_NAME}' (Anchors=Gold, Edges=Cyan)", color='white', fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# 2. INTERACTIVE PLOTLY HTML EXPORT (For rotatable WebGL viewing in browser)
# ------------------------------------------------------------------------------
html_file_scales = "./outputs/wireframe_scales_comparison.html"
wire_x, wire_y, wire_z = [], [], []
for i, j in edges:
    wire_x.extend([anchors[i, 0], anchors[j, 0], None])
    wire_y.extend([anchors[i, 1], anchors[j, 1], None])
    wire_z.extend([anchors[i, 2], anchors[j, 2], None])

fig_plotly = make_subplots(
    rows=1, cols=4,
    specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
    subplot_titles=[f"N = {N}" for N in SCALE_COUNTS],
    horizontal_spacing=0.02
)

for col_idx, N in enumerate(SCALE_COUNTS, start=1):
    points, point_types = build_swarm_from_graph(graph, N)
    fig_plotly.add_trace(go.Scatter3d(x=wire_x, y=wire_y, z=wire_z, mode="lines", line=dict(color="rgba(255,255,255,0.22)", width=1.5, dash="dash"), hoverinfo="none", showlegend=False), row=1, col=col_idx)
    anchor_mask = [t == "anchor" for t in point_types]
    if any(anchor_mask):
        apts = points[anchor_mask]
        fig_plotly.add_trace(go.Scatter3d(x=apts[:,0], y=apts[:,1], z=apts[:,2], mode="markers", marker=dict(size=6, color="#FFD700"), name="Anchors", showlegend=(col_idx==1)), row=1, col=col_idx)
    edge_mask = [t == "edge" for t in point_types]
    if any(edge_mask):
        epts = points[edge_mask]
        fig_plotly.add_trace(go.Scatter3d(x=epts[:,0], y=epts[:,1], z=epts[:,2], mode="markers", marker=dict(size=4, color="#00E5FF"), name="Edges", showlegend=(col_idx==1)), row=1, col=col_idx)

scene_cube = dict(aspectmode="cube", xaxis=dict(range=[-1.1, 1.1], backgroundcolor="#0b0f19"), yaxis=dict(range=[-1.1, 1.1], backgroundcolor="#0b0f19"), zaxis=dict(range=[-1.1, 1.1], backgroundcolor="#0b0f19"))
fig_plotly.update_layout(template="plotly_dark", height=400, scene=scene_cube, scene2=scene_cube, scene3=scene_cube, scene4=scene_cube, margin=dict(l=10, r=10, b=10, t=50))
fig_plotly.write_html(html_file_scales)

display(HTML(f"""
<div style="margin-top:8px; padding:10px; background-color:#1e293b; border-radius:6px; border-left:4px solid #FFD700;">
    <span style="color:#94a3b8; font-size:12px;">INTERACTIVE 3D WEBGL VIEWER:</span><br>
    <a style="color:#38bdf8; font-weight:bold; font-size:13px; text-decoration:underline;" href="{html_file_scales}" target="_blank">
        Click here to open interactive rotatable 3D file: {os.path.basename(html_file_scales)}
    </a>
</div>
"""))

## Cell 4: All Shapes Gallery (`fighter_jet`, `star_3d`, `chair`, `house_3d`)

Renders a $2 \times 2$ **True 3D Perspective Grid** displaying all 4 shapes at $N = 35$ drones:
- **Top-Left:** `fighter_jet` (Swept wings with twin vertical tail fins)
- **Top-Right:** `star_3d` (5-pointed volumetric star with apex hubs)
- **Bottom-Left:** `chair` (4 legs, seat frame, tall backrest)
- **Bottom-Right:** `house_3d` (Gable roof and rectangular walls)
- **Zero WebGL Dependency:** Renders inline directly into the notebook cell.

In [ ]:
# ==============================================================================
# Cell 4: All Shapes Gallery (fighter_jet, star_3d, chair, house_3d)
# ==============================================================================
NUM_DRONES_GALLERY = 35
all_shapes = list(wireframe_db.keys())

# ------------------------------------------------------------------------------
# 1. TRUE 3D MATPLOTLIB 2x2 GALLERY (100% Visible - No WebGL dependency)
# ------------------------------------------------------------------------------
fig_gallery_mpl = plt.figure(figsize=(14, 12))
fig_gallery_mpl.patch.set_facecolor('#0b0f19')

for idx, shape_name in enumerate(all_shapes, start=1):
    ax = fig_gallery_mpl.add_subplot(2, 2, idx, projection='3d')
    ax.set_facecolor('#0b0f19')

    g = wireframe_db[shape_name]
    anchors = np.array(g["anchors"], dtype=float)
    edges = g["edges"]

    points, point_types = build_swarm_from_graph(g, NUM_DRONES_GALLERY)
    d_min = compute_min_distance(points)

    # 1. Wireframe edges
    for i, j in edges:
        ax.plot(
            [anchors[i, 0], anchors[j, 0]],
            [anchors[i, 1], anchors[j, 1]],
            [anchors[i, 2], anchors[j, 2]],
            color='white', linestyle='--', linewidth=0.8, alpha=0.25
        )

    # 2. Anchors (Gold)
    anchor_mask = [t == "anchor" for t in point_types]
    if any(anchor_mask):
        apts = points[anchor_mask]
        ax.scatter(apts[:, 0], apts[:, 1], apts[:, 2], c='#FFD700', s=80, edgecolors='white', linewidth=0.8, depthshade=True)

    # 3. Edge drones (Cyan)
    edge_mask = [t == "edge" for t in point_types]
    if any(edge_mask):
        epts = points[edge_mask]
        ax.scatter(epts[:, 0], epts[:, 1], epts[:, 2], c='#00E5FF', s=40, depthshade=True)

    ax.set_title(f"{shape_name.upper()}\n(N = {NUM_DRONES_GALLERY}, d_min = {d_min:.3f})", color='white', fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel("X", color='#94a3b8', labelpad=4)
    ax.set_ylabel("Y", color='#94a3b8', labelpad=4)
    ax.set_zlabel("Z", color='#94a3b8', labelpad=4)
    ax.tick_params(colors='#94a3b8', labelsize=8)
    ax.set_box_aspect([1, 1, 1])
    ax.view_init(elev=22, azim=45)

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('#1e293b')
    ax.yaxis.pane.set_edgecolor('#1e293b')
    ax.zaxis.pane.set_edgecolor('#1e293b')
    ax.grid(color='#334155', linestyle=':', alpha=0.5)

plt.suptitle(f"3D Wireframe Drone Swarm Gallery (N = {NUM_DRONES_GALLERY} Drones Each)", color='white', fontsize=16, fontweight='bold', y=0.96)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# 2. INTERACTIVE PLOTLY HTML EXPORT (For rotatable WebGL viewing in browser)
# ------------------------------------------------------------------------------
html_file_gallery = "./outputs/wireframe_gallery_comparison.html"
fig_gallery_plotly = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}], [{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=[f"{s.upper()} (N={NUM_DRONES_GALLERY})" for s in all_shapes],
    horizontal_spacing=0.04, vertical_spacing=0.08
)

for (row, col), shape_name in zip([(1, 1), (1, 2), (2, 1), (2, 2)], all_shapes):
    g = wireframe_db[shape_name]
    anchors = np.array(g["anchors"], dtype=float)
    edges = g["edges"]
    wx, wy, wz = [], [], []
    for i, j in edges:
        wx.extend([anchors[i, 0], anchors[j, 0], None])
        wy.extend([anchors[i, 1], anchors[j, 1], None])
        wz.extend([anchors[i, 2], anchors[j, 2], None])

    fig_gallery_plotly.add_trace(go.Scatter3d(x=wx, y=wy, z=wz, mode="lines", line=dict(color="rgba(255,255,255,0.22)", width=1.5, dash="dash"), hoverinfo="none", showlegend=False), row=row, col=col)
    points, point_types = build_swarm_from_graph(g, NUM_DRONES_GALLERY)
    anchor_mask = [t == "anchor" for t in point_types]
    if any(anchor_mask):
        apts = points[anchor_mask]
        fig_gallery_plotly.add_trace(go.Scatter3d(x=apts[:,0], y=apts[:,1], z=apts[:,2], mode="markers", marker=dict(size=6, color="#FFD700"), showlegend=((row,col)==(1,1)), name="Anchors"), row=row, col=col)
    edge_mask = [t == "edge" for t in point_types]
    if any(edge_mask):
        epts = points[edge_mask]
        fig_gallery_plotly.add_trace(go.Scatter3d(x=epts[:,0], y=epts[:,1], z=epts[:,2], mode="markers", marker=dict(size=4.5, color="#00E5FF"), showlegend=((row,col)==(1,1)), name="Edges"), row=row, col=col)

scene_g_cube = dict(aspectmode="cube", xaxis=dict(range=[-1.1, 1.1], backgroundcolor="#0b0f19"), yaxis=dict(range=[-1.1, 1.1], backgroundcolor="#0b0f19"), zaxis=dict(range=[-1.1, 1.1], backgroundcolor="#0b0f19"))
fig_gallery_plotly.update_layout(template="plotly_dark", height=750, scene=scene_g_cube, scene2=scene_g_cube, scene3=scene_g_cube, scene4=scene_g_cube, margin=dict(l=10, r=10, b=10, t=50))
fig_gallery_plotly.write_html(html_file_gallery)

display(HTML(f"""
<div style="margin-top:8px; padding:10px; background-color:#1e293b; border-radius:6px; border-left:4px solid #00E5FF;">
    <span style="color:#94a3b8; font-size:12px;">INTERACTIVE 3D WEBGL GALLERY:</span><br>
    <a style="color:#38bdf8; font-weight:bold; font-size:13px; text-decoration:underline;" href="{html_file_gallery}" target="_blank">
        Click here to open interactive rotatable 3D gallery: {os.path.basename(html_file_gallery)}
    </a>
</div>
"""))